# DUNE FABRIC Live Visualization Implementation Notebook

This notebook is the front door for the packaged Queue v2 `dune_fabric_live_demo` tool. A new user should be able to understand the first run path from this notebook alone. The supporting docs are linked later as references, not prerequisites.

## What This Tool Does

- Runs the packaged DUNE/FABRIC live visualization pipeline from YAML config.
- Supports `pcap_replay` mode now as the controlled and validated test mode.
- Provides a `live_nic` direction for real data, where packets already arrive on the receiver NIC.
- Preserves raw PCAP chunks, decodes them into CSV chunks, writes manifest/status reports, and renders heatmap plus ridgeline visualizations.
- Keeps the transport backend separate from the pipeline: LoomAI is currently validated, and SSH is the portability path for external/local Jupyter.

## System Architecture

```text
pcap_replay mode: sender VM tcpreplay selected PCAP
                                             |
live_nic mode: real traffic already on receiver NIC
                                             |
                                             v
receiver NIC -> raw PCAP chunks -> queue -> decoded CSV chunks -> manifest/status -> visualization
```

PCAP replay mode is the controlled test mode. It uses `tcpreplay` on the sender VM. Live NIC mode is the real-data direction. It listens on the receiver NIC and skips `tcpreplay`. After packets reach the receiver, both modes should use the same Queue v2 path shown above.

## Execution Map: Local Machine vs FABRIC VMs

| Location | What runs there | Required for first run |
| --- | --- | --- |
| Local/external Jupyter machine | This notebook, `scripts/run_pipeline.py`, dry-run planning, final visualization rendering, local output directory | Python with `pyyaml`; `pandas`/`matplotlib` for visualization; `ssh`/`scp` or `rsync` for SSH transport |
| FABRIC sender VM | `tcpreplay` for `pcap_replay`; source PCAP chunking/staging | Sender management address, data-plane interface, `tcpreplay`, source PCAP file |
| FABRIC receiver VM | `tcpdump`, raw PCAP chunk preservation, Queue v2 decode/export, validation helper, combined CSV generation | Receiver management address, data-plane interface, `tcpdump`, Python/WIBViz code, mapping CSV, optional reference CSV |

Local outputs default to `/tmp/dune_fabric_live_demo/runs/<run_id>/` on the machine running Jupyter, not necessarily on a FABRIC VM. Important local files include `reports/manifest.json`, `artifacts/combined.csv`, `visualizations/heatmap.png`, and `visualizations/stacked_waveform.png`. Do not write large runtime outputs into `/home/fabric/work`; disk space there is limited.

Remote outputs are written under the configured VM roots, normally `/home/ubuntu/wibviz_runs/dune_fabric_live_demo/<run_id>/` and `/home/ubuntu/wibviz_outputs/dune_fabric_live_demo/<run_id>/` on the receiver VM.

## Current Validation Status

LoomAI transport is currently validated with event 4245 in `pcap_replay` mode. The full validated run was `20260716T185535Z-event-4245-pcap_replay`: 10 raw chunks preserved, 409,600 decoded rows, validation passed with 0 missing / 0 extra, and both heatmap and ridgeline were generated.

SSH transport is implemented and dry-run validated. A real SSH smoke test still requires actual VM hostnames/IPs, SSH key/config, and VM-side input files.

## What Must Already Exist

Before a real smoke test or full run, confirm this checklist:

- Active FABRIC slice.
- Sender VM reachable for `pcap_replay` mode.
- Receiver VM reachable.
- Detector/data interface name known on the sender and receiver VMs.
- `tcpdump` installed on the receiver VM, with non-interactive permission for capture.
- `tcpreplay` installed on the sender VM for replay mode.
- Python dependencies available on the receiver VM for the WIBViz decoder/exporter.
- PCAP file already present on the sender VM for replay mode.
- Mapping CSV already present on the receiver VM.
- Optional reference CSV already present on the receiver VM when validation is enabled.
- SSH credentials/config available if using external/local Jupyter with `transport.type: ssh`.

## What This Notebook Can Automate Now vs What Remains Manual

| Task | Status | Notes |
| --- | --- | --- |
| Runs pipeline from config | Automated | Uses `scripts/run_pipeline.py`. |
| Dry-run/preflight plan | Automated | Dry-run does not contact VMs. Real preflight runs at smoke/full time. |
| Bounded smoke test | Automated | Requires VM reachability and required files/tools. |
| Full replay | Automated with confirmation | Controlled by `run_full_replay = True`; do not use until smoke passes. |
| Visualization display | Automated | Reads local `heatmap.png` and `stacked_waveform.png` after a real run. |
| Slice creation | Not fully automated yet | Create/provision the FABRIC slice outside this notebook for now. |
| VM dependency installation | Partially/manual | The runner checks tools but does not install all dependencies. |
| Sender/receiver file staging | Manual | PCAPs, mapping CSV, reference CSV, and WIBViz source must already be on VMs. |
| SSH host/key discovery | Manual for now | Fill SSH templates from your FABRIC slice details. |
| Live continuous stream | Prototype path | Use bounded chunks first; unbounded `--follow` is not polished production mode. |

## Step 0: Choose Execution Mode

Choose the environment first, then choose the input mode.

- If you are running inside LoomAI, use `transport_backend = "loom"`. This is the currently validated path.
- If you are running from external/local Jupyter, use `transport_backend = "ssh"`, copy an SSH template to a real config file, and fill in the connection details below.

Then choose input mode:

- `source_mode = "pcap_replay"`: controlled replay from a PCAP on the sender VM.
- `source_mode = "live_nic"`: real incoming packets already arriving on the receiver VM NIC.

The selection code cell below defaults to the validated LoomAI event 4245 replay and does not contact VMs.

## If You Are Running Inside LoomAI

Use this path when this notebook is open in the LoomAI-managed Jupyter/container environment.

What you need to know:

- Keep `transport_backend = "loom"`.
- The runner uses `loomai exec` and `loomai scp` internally through `scripts/remote_transport.py`.
- `slice.name`, `slice.sender_node`, and `slice.receiver_node` must match an existing LoomAI-visible FABRIC slice.
- The validated baseline config is `configs/event_4245_replay.yaml`.
- Local notebook outputs still go under `/tmp/dune_fabric_live_demo/runs/<run_id>/` unless `output_root_override` is set.

First safe command in this path:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --dry-run
```

First real smoke-test command after the dry-run looks correct:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --max-live-chunks 2
```

## If You Are Running From External/Local Jupyter

Use this path from a laptop, desktop, or standard FABRIC Jupyter environment that is not inside LoomAI.

What you need locally:

- A copy or clone of this `dune_fabric_live_demo` package.
- Python dependencies for this notebook/runner: `pyyaml`, and for visualization after real runs, `pandas` and `matplotlib`.
- Local command-line SSH tools: `ssh` plus `scp`, or `ssh` plus `rsync` if you set `copy_tool: rsync`.
- Network reachability to FABRIC VM management addresses, usually through a FABRIC SSH config or bastion/ProxyJump.

What you need from the FABRIC slice:

- Sender VM management IP/DNS/SSH config alias for `pcap_replay`.
- Receiver VM management IP/DNS/SSH config alias.
- SSH username, usually `ubuntu` for the packaged examples.
- Private key path or SSH config containing the identity file.
- Data-plane interface names inside the VMs, for example `enp7s0`.

External/local Jupyter first-run sequence:

1. Copy `configs/event_4245_replay_ssh_template.yaml` to `configs/event_4245_replay_ssh.yaml`.
2. Fill in the SSH config fields described below.
3. Set `transport_backend = "ssh"` and `config_path_override = "configs/event_4245_replay_ssh.yaml"` in the selection cell.
4. Run the dry-run cell.
5. Use the optional SSH access-check cell to verify benign remote commands.
6. Run the 2-chunk smoke test only after the access check and required VM files are correct.

## External Jupyter SSH Setup Helper

Use this helper if you are not already comfortable with SSH. The LoomAI path is validated. The SSH path is implemented and dry-run validated, but a true external SSH smoke test still requires real VM hosts, keys, and staged VM files.

### 1. What You Need

| Item | Why it matters |
| --- | --- |
| Local machine or external Jupyter environment | Runs this notebook and `scripts/run_pipeline.py`. |
| FABRIC account/access | Needed to use the slice VMs. |
| Active FABRIC slice with sender and receiver VMs | The runner does not create the slice yet. |
| Sender/receiver management IPs or hostnames | Used by SSH; these are not detector-net IPs. |
| SSH private key accepted by FABRIC | Used to log in to the VMs. |
| Package directory copied/cloned locally | Contains this notebook, configs, and scripts. |
| Required files already staged on VMs | PCAP on sender; mapping/reference/WIBViz files on receiver. |

### 2. Check Whether You Already Have An SSH Key

From a terminal on the local/Jupyter machine:

```bash
ls ~/.ssh
ls -la ~/.ssh
```

Common key pairs look like this:

| Private key | Public key |
| --- | --- |
| `~/.ssh/id_ed25519` | `~/.ssh/id_ed25519.pub` |
| `~/.ssh/id_rsa` | `~/.ssh/id_rsa.pub` |
| `~/.ssh/fabric_wibviz` | `~/.ssh/fabric_wibviz.pub` |

The private key has no `.pub` suffix. Do not share the private key.

### 3. Generate A New SSH Key If Needed

If you do not already have a FABRIC-accepted key, generate one:

```bash
ssh-keygen -t ed25519 -f ~/.ssh/fabric_wibviz -C "fabric-wibviz"
```

This creates two files. `~/.ssh/fabric_wibviz` is the private key and stays on your local/Jupyter machine. `~/.ssh/fabric_wibviz.pub` is the public key and is what you register or share for access.

### 4. Register The Public Key For FABRIC Use

The exact registration step depends on your FABRIC/project setup. Add the public key wherever FABRIC SSH keys are managed, or use the key your project already provides.

Print the public key with:

```bash
cat ~/.ssh/fabric_wibviz.pub
```

Do not paste or upload `~/.ssh/fabric_wibviz`; that is the private key.

### 5. Find Sender And Receiver Management Hosts

Use the management IPs, DNS names, or SSH config aliases for the VMs. Do not use detector/data-plane addresses such as `192.168.1.1` or `192.168.1.2` for SSH unless your environment explicitly routes them for management.

```bash
SENDER_HOST=<sender-management-host-or-ip>
RECEIVER_HOST=<receiver-management-host-or-ip>
```

### 6. Test Raw SSH Access Manually

For the packaged Ubuntu examples:

```bash
ssh -i ~/.ssh/fabric_wibviz ubuntu@$SENDER_HOST hostname
ssh -i ~/.ssh/fabric_wibviz ubuntu@$RECEIVER_HOST hostname
```

If your site requires a bastion or ProxyJump:

```bash
ssh -i ~/.ssh/fabric_wibviz -J <bastion-user>@<bastion-host> ubuntu@$SENDER_HOST hostname
```

If these commands fail, the notebook pipeline will not work yet. Fix raw SSH first.

### 7. Create The Package SSH Config

Copy the template:

```bash
cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml
```

Edit the copied file. The SSH user field is named `username` in this package config.

Example:

```yaml
transport:
  type: ssh
  username: ubuntu
  sender_host: 198.51.100.10
  receiver_host: 198.51.100.11
  ssh_key: ~/.ssh/fabric_wibviz
  ssh_options:
    - -o
    - BatchMode=yes
    - -o
    - StrictHostKeyChecking=accept-new
  copy_tool: scp
```

If you need ProxyJump in the package config, add it to `ssh_options`:

```yaml
ssh_options:
  - -J
  - <bastion-user>@<bastion-host>
  - -o
  - BatchMode=yes
```

Also verify the remote file paths in the same YAML: `pcap_replay.sender_pcap` on the sender VM, `paths.mapping_csv` on the receiver VM, and optional `validation.reference_csv` on the receiver VM.

### 8. Test Package-Level SSH Dry-Run

Dry-run parses the config and shows planned commands. It does not contact VMs or run replay.

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --dry-run
```

### 9. Run The First SSH Smoke Test

Only after dry-run and raw SSH access work, run the two-chunk smoke test:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --max-live-chunks 2
```

### 10. SSH Setup Troubleshooting

| Symptom | Likely cause | First fix |
| --- | --- | --- |
| `Permission denied (publickey)` | Wrong key/user or public key not registered | Check `username`, `ssh_key`, and FABRIC/project key registration. |
| `Connection timed out` | Wrong host, VPN/bastion/security-group issue | Verify management host and any required bastion/ProxyJump. |
| `No route to host` | Local environment cannot reach the FABRIC management network | Use the FABRIC SSH config, VPN, or correct bastion path. |
| `tcpreplay: not found` | Sender VM dependency missing | Install `tcpreplay` on the sender VM. |
| `tcpdump: not found` | Receiver VM dependency missing | Install `tcpdump` on the receiver VM. |
| PCAP not found | Source PCAP missing on sender | Stage the PCAP at `pcap_replay.sender_pcap`. |
| Mapping CSV not found | Mapping CSV missing on receiver | Stage the mapping CSV at `paths.mapping_csv`. |

## Connection Details You Need

Collect these before editing an SSH config:

| Field | Meaning | Example |
| --- | --- | --- |
| `transport.username` | VM login user | `ubuntu` |
| `transport.sender_host` | Sender VM management IP, DNS name, or SSH config alias | `192.0.2.10` or `sender` |
| `transport.receiver_host` | Receiver VM management IP, DNS name, or SSH config alias | `192.0.2.11` or `receiver` |
| `transport.ssh_key` | Private key path on the local/Jupyter machine | `~/.ssh/fabric_slice_key` |
| `transport.ssh_options` | Extra `ssh` arguments before `user@host` | `-F /path/to/fabric_ssh_config` or `-J bastion` |
| `network.sender_interface` | Sender data-plane NIC for `tcpreplay` | `enp7s0` |
| `network.receiver_interface` | Receiver data-plane NIC for `tcpdump` | `enp7s0` |
| `pcap_replay.sender_pcap` | Absolute PCAP path on the sender VM | `/home/ubuntu/dune-replay/pcap/event.pcap` |
| `paths.mapping_csv` | Absolute mapping CSV path on the receiver VM | `/home/ubuntu/wibviz_inputs/event-4245-channel-map.csv` |
| `validation.reference_csv` | Optional absolute reference CSV path on the receiver VM | `/home/ubuntu/wibviz_inputs/4245.csv` |

If you use a FABRIC-generated SSH config, `sender_host` and `receiver_host` may be `Host` aliases from that file instead of raw IP addresses.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# External/local Jupyter users: start Jupyter from this package directory, or set
# package_root_override to the cloned/copied dune_fabric_live_demo path.
package_root_override = ""  # e.g. "/Users/me/dune_fabric_live_demo"
if package_root_override:
    PACKAGE_ROOT = Path(package_root_override).expanduser().resolve()
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    PACKAGE_ROOT = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "scripts/run_pipeline.py").is_file()
            and (candidate / "configs").is_dir()
        ),
        None,
    )
    if PACKAGE_ROOT is None:
        raise RuntimeError(
            "Could not find dune_fabric_live_demo. Start Jupyter from the package "
            "directory or set package_root_override."
        )

# Choose the pipeline input and transport.
source_mode = "pcap_replay"   # "pcap_replay" or "live_nic"
demo = "event_4245"           # loom: event_4245, event_6860, custom_pcap, live_nic; ssh: event_4245 or live_nic
transport_backend = "loom"    # "loom" = currently validated; "ssh" = external/local Jupyter portability path

# Optional explicit config. Use this after copying an SSH template, for example:
# config_path_override = "configs/event_4245_replay_ssh.yaml"
config_path_override = ""

# Safety controls for cells below.
smoke_chunks = 2               # bounded smoke-test chunks; also bounds pcap_replay via --max-live-chunks
run_smoke_test = False         # set True only after dry-run paths/config look correct
run_full_replay = True        # set True only when ready for the complete pcap replay
enable_adaptive = False        # experimental; keep False for the validated Queue v2 baseline
validate_results = True        # set False to pass --no-validate
output_root_override = ""      # optional local output root; keep large outputs out of /home/fabric/work

config_map = {
    "loom": {
        "event_4245": PACKAGE_ROOT / "configs/event_4245_replay.yaml",
        "event_6860": PACKAGE_ROOT / "configs/event_6860_replay.yaml",
        "custom_pcap": PACKAGE_ROOT / "configs/custom_pcap_replay_template.yaml",
        "live_nic": PACKAGE_ROOT / "configs/live_nic_template.yaml",
    },
    "ssh": {
        "event_4245": PACKAGE_ROOT / "configs/event_4245_replay_ssh_template.yaml",
        "live_nic": PACKAGE_ROOT / "configs/live_nic_ssh_template.yaml",
    },
}

if transport_backend not in config_map:
    raise ValueError(f"unknown transport_backend: {transport_backend}")
selected_demo = "live_nic" if source_mode == "live_nic" else demo
if selected_demo not in config_map[transport_backend]:
    raise ValueError(f"{transport_backend} transport has no config template for {selected_demo}")

if config_path_override:
    candidate = Path(config_path_override).expanduser()
    config_path = candidate if candidate.is_absolute() else PACKAGE_ROOT / candidate
else:
    config_path = config_map[transport_backend][selected_demo]

if not config_path.is_file():
    raise FileNotFoundError(f"Config does not exist: {config_path}")

print(f"Package:   {PACKAGE_ROOT}")
print(f"Config:    {config_path}")
print(f"Mode:      {source_mode}")
print(f"Transport: {transport_backend}")
if output_root_override:
    print(f"Local output override: {Path(output_root_override).expanduser()}")
if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
    print("SSH template selected. Dry-run can inspect it, but copy/edit it before smoke or full runs.")

## Step 1: Prepare Or Identify FABRIC Slice

This notebook does not create or provision the FABRIC slice yet. Start with a slice that already has the sender/receiver topology you want to use.

For LoomAI mode, the config uses `slice.name`, `slice.sender_node`, and `slice.receiver_node` so LoomAI can resolve VM access.

For SSH mode, `slice.name` is informational. The logical node names still matter because the transport maps `slice.sender_node` and `slice.receiver_node` to `transport.sender_host` and `transport.receiver_host`.

## Optional: FABRIC Slice Setup With FABlib

This section is optional and state-changing if you enable creation/submission. It is included to help external/local Jupyter users get from a FABRIC slice to the SSH config values needed by this demo. Nothing here runs automatically.

Recommended safe path:

1. Use an existing FABRIC slice if you already have one.
2. Identify the sender and receiver nodes in that slice.
3. Get each node's management IP or SSH hostname for `transport.sender_host` and `transport.receiver_host`.
4. Confirm the data-plane NIC names for `network.sender_interface` and `network.receiver_interface`.
5. Install VM dependencies and stage files separately.
6. Copy and edit `configs/event_4245_replay_ssh_template.yaml`.

New-slice scaffold path:

- The scaffold below shows a two-node sender/receiver topology with one `NIC_Basic` on each node and an L2 experiment network between them.
- It does not submit by default. Submission allocates real FABRIC resources and should only be enabled intentionally.
- After a submitted slice becomes ready, use the management IPs as SSH hosts. Do not use detector/data-plane addresses like `192.168.1.1` and `192.168.1.2` for SSH unless your environment explicitly routes them for management.

Dependency installation still remains manual for now: `tcpreplay` and the PCAP on the sender; `tcpdump`, Python/WIBViz, mapping CSV, and optional reference CSV on the receiver.

In [ ]:
# Optional FABlib setup scaffold. Safe defaults: no FABRIC state changes.
import importlib.util

run_fablib_preflight = False       # True: instantiate FablibManager and inspect availability/config
use_existing_fabric_slice = True   # True: inspect existing slice; False: build a new in-memory topology scaffold
run_fablib_create_draft = False    # True with use_existing_fabric_slice=False: build local topology object
run_fablib_submit = False          # True: submit the new slice; this allocates FABRIC resources

fablib_slice_name = "dune-wibviz-demo"
fablib_site = "RENC"              # Change to a site with available resources
fablib_image = "default_ubuntu_22"
fablib_sender_node = "sender"
fablib_receiver_node = "receiver"
fablib_cores = 4
fablib_ram = 16
fablib_disk = 50
fablib_network_name = "dune-data-net"

fablib_spec = importlib.util.find_spec("fabrictestbed_extensions.fablib.fablib")
print("FABlib import available:", bool(fablib_spec))
print("Planned slice:", fablib_slice_name)
print("Sender node:", fablib_sender_node)
print("Receiver node:", fablib_receiver_node)
print("Site:", fablib_site)
print("State-changing flags:", {
    "run_fablib_preflight": run_fablib_preflight,
    "run_fablib_create_draft": run_fablib_create_draft,
    "run_fablib_submit": run_fablib_submit,
})

if not fablib_spec:
    print("FABlib is not installed in this environment. Use LoomAI slice tools or run this from a FABRIC/FABlib environment.")
elif not (run_fablib_preflight or run_fablib_create_draft or run_fablib_submit):
    print("No FABlib calls were made. Set one of the run_fablib_* flags above when you intentionally want to inspect or create slice state.")
else:
    from fabrictestbed_extensions.fablib.fablib import FablibManager

    fablib = FablibManager()
    print("FablibManager initialized.")

    if use_existing_fabric_slice:
        if not run_fablib_preflight:
            print("Set run_fablib_preflight=True to inspect an existing slice.")
        else:
            fabric_slice = fablib.get_slice(name=fablib_slice_name)
            print("Existing slice:", fabric_slice.get_name(), fabric_slice.get_state())
            for node_name in (fablib_sender_node, fablib_receiver_node):
                node = fabric_slice.get_node(name=node_name)
                management_ip = node.get_management_ip()
                print(f"{node_name} management host for SSH config: {management_ip}")
            print("Use these as transport.sender_host and transport.receiver_host in the SSH YAML.")
    else:
        if not (run_fablib_create_draft or run_fablib_submit):
            print("Set run_fablib_create_draft=True to build the local topology scaffold.")
        else:
            fabric_slice = fablib.new_slice(name=fablib_slice_name)
            sender = fabric_slice.add_node(
                name=fablib_sender_node,
                site=fablib_site,
                cores=fablib_cores,
                ram=fablib_ram,
                disk=fablib_disk,
                image=fablib_image,
            )
            receiver = fabric_slice.add_node(
                name=fablib_receiver_node,
                site=fablib_site,
                cores=fablib_cores,
                ram=fablib_ram,
                disk=fablib_disk,
                image=fablib_image,
            )
            sender_nic = sender.add_component(model="NIC_Basic", name="nic1")
            receiver_nic = receiver.add_component(model="NIC_Basic", name="nic1")
            fabric_slice.add_l2network(
                name=fablib_network_name,
                interfaces=[sender_nic.get_interfaces()[0], receiver_nic.get_interfaces()[0]],
            )
            print("Built local two-node FABlib topology scaffold. No resources allocated yet.")
            if run_fablib_submit:
                print("Submitting slice. This is state-changing and may take several minutes.")
                fabric_slice.submit()
                fabric_slice.wait_ssh(progress=True)
                for node_name in (fablib_sender_node, fablib_receiver_node):
                    node = fabric_slice.get_node(name=node_name)
                    print(f"{node_name} management host for SSH config: {node.get_management_ip()}")
            else:
                print("run_fablib_submit=False, so the slice was not submitted.")

## Step 2: Identify Sender And Receiver VMs

For `pcap_replay`, identify both VMs:

- Sender VM: holds the source PCAP and runs `tcpreplay` on `network.sender_interface`.
- Receiver VM: runs `tcpdump`, preserves raw chunks, decodes chunks, validates output, and provides the combined CSV for local visualization.

For `live_nic`, the sender VM is not used by the runner. The receiver VM must already be receiving real packets on `network.receiver_interface`.

For external/local Jupyter, record the management address or SSH config alias for each VM. If FABRIC access requires a bastion, use the FABRIC-generated SSH config or equivalent `ProxyJump` settings in `transport.ssh_options`.

## Step 3: Prepare VM Dependencies

The runner performs preflight checks during real smoke/full runs, but it does not install everything automatically.

Receiver VM requirements:

- `python3`
- `tcpdump`
- non-interactive capture permission, usually passwordless `sudo tcpdump`
- WIBViz source/exporter at `paths.wibviz_src`
- mapping CSV at `paths.mapping_csv`
- optional reference CSV if `validation.enabled: true`

Sender VM requirements for `pcap_replay`:

- `tcpreplay`
- source PCAP at `pcap_replay.sender_pcap`
- configured data-plane interface at `network.sender_interface`

## Step 4: Required VM Files

Paths in the YAML config are VM paths unless explicitly described as local outputs. The runner uploads only its helper validation script. It does not upload the PCAP, mapping CSV, reference CSV, or WIBViz source tree.

Required on the sender VM for `pcap_replay`:

- Source PCAP at `pcap_replay.sender_pcap`.
- `tcpreplay` installed and runnable.
- Data-plane interface from `network.sender_interface` exists.

Required on the receiver VM for both modes:

- `tcpdump` installed and usable non-interactively, usually via `sudo tcpdump`.
- `python3` installed.
- WIBViz source/exporter at `paths.wibviz_src`.
- Mapping CSV at `paths.mapping_csv`.
- Data-plane interface from `network.receiver_interface` exists.

Required on the receiver VM only when validation is enabled:

- Reference CSV at `validation.reference_csv`.
- `validation.compare_fields` columns present in both generated output and reference CSV.

Local to the Jupyter machine:

- `paths.output_root` is where dry-run reports, downloaded `combined.csv`, and visualization PNGs are written.
- Keep large local outputs in `/tmp` or local scratch, not in a small shared workspace.

## Step 5: Choose Config File

Available starting points:

- `configs/event_4245_replay.yaml`: validated LoomAI PCAP replay baseline.
- `configs/event_6860_replay.yaml`: LoomAI PCAP replay for event 6860.
- `configs/custom_pcap_replay_template.yaml`: LoomAI template for a different PCAP.
- `configs/live_nic_template.yaml`: LoomAI live NIC template.
- `configs/event_4245_replay_ssh_template.yaml`: external/local Jupyter SSH template for event 4245 replay.
- `configs/live_nic_ssh_template.yaml`: external/local Jupyter SSH template for live NIC mode.

For SSH, copy a template before editing it:

```bash
cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml
cp configs/live_nic_ssh_template.yaml configs/live_nic_ssh.yaml
```

Then set `config_path_override` in Step 0 to the copied config.

## How To Fill In The SSH Config

For external/local Jupyter, never run smoke/full replay against the `_template.yaml` file. Copy it first:

```bash
cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml
```

Then edit the copied file. Minimum replay config fields:

```yaml
transport:
  type: ssh
  username: ubuntu
  sender_host: <sender-management-ip-or-ssh-config-alias>
  receiver_host: <receiver-management-ip-or-ssh-config-alias>
  ssh_key: ~/.ssh/fabric_wibviz
  ssh_options:
    - -o
    - BatchMode=yes
  copy_tool: scp

network:
  sender_interface: enp7s0
  receiver_interface: enp7s0

paths:
  output_root: /tmp/dune_fabric_live_demo      # local to Jupyter
  receiver_root: /home/ubuntu                  # receiver VM
  sender_root: /home/ubuntu                    # sender VM
  wibviz_src: /home/ubuntu/wibviz_graph/src    # receiver VM
  mapping_csv: /home/ubuntu/wibviz_inputs/event-4245-channel-map.csv

pcap_replay:
  sender_pcap: /home/ubuntu/dune-replay/pcap/icecosmich_run82134054_sub001_evt004245.pcap
```

If your key is already specified by a FABRIC SSH config, leave `ssh_key: ""` and add:

```yaml
ssh_options:
  - -F
  - /path/to/fabric_ssh_config
  - -o
  - BatchMode=yes
```

If you need a bastion directly, use `ProxyJump`/`-J`:

```yaml
ssh_options:
  - -J
  - <bastion-user>@<bastion-host>
  - -o
  - BatchMode=yes
```

The logical names in `slice.sender_node` and `slice.receiver_node` must map to SSH hosts. With the packaged template, `sender_node: sender` maps to `transport.sender_host`, and `receiver_node: receiver` maps to `transport.receiver_host`.

In [ ]:
import yaml

with config_path.open(encoding="utf-8") as handle:
    selected_config = yaml.safe_load(handle)

important = {
    "label": selected_config.get("label"),
    "source_mode": selected_config.get("source_mode"),
    "transport": selected_config.get("transport"),
    "slice": selected_config.get("slice"),
    "network": selected_config.get("network"),
    "paths": selected_config.get("paths"),
    "pcap_replay": selected_config.get("pcap_replay"),
    "live_nic": selected_config.get("live_nic"),
    "validation": selected_config.get("validation"),
}
print(json.dumps(important, indent=2, sort_keys=True))

serialized = json.dumps(important)
if "replace-with-" in serialized:
    print("\nWARNING: This config still contains placeholder host values. Use dry-run only until copied and edited.")

transport = selected_config.get("transport", {})
paths = selected_config.get("paths", {})
print("\nPath interpretation:")
print(f"- Local run output root: {paths.get('output_root')} on the Jupyter/local machine running this notebook")
print(f"- Receiver paths are on the receiver VM, rooted at: {paths.get('receiver_root')}")
print(f"- Sender paths are on the sender VM, rooted at: {paths.get('sender_root')}")
if transport.get("type") == "ssh":
    print("- SSH host/key fields must be reachable from this Jupyter environment before smoke/full runs.")

## How To Check VM Access

Dry-run validates config shape and planned commands without contacting VMs. Before a smoke test from external/local Jupyter, verify SSH with harmless commands.

Manual checks from a terminal look like this:

```bash
ssh -i ~/.ssh/fabric_slice_key ubuntu@<receiver-host> 'hostname; whoami; command -v python3; command -v tcpdump; ip -brief addr'
ssh -i ~/.ssh/fabric_slice_key ubuntu@<sender-host> 'hostname; whoami; command -v tcpreplay; ip -brief addr'
```

If using a FABRIC SSH config:

```bash
ssh -F /path/to/fabric_ssh_config ubuntu@<receiver-alias> 'hostname; whoami; ip -brief addr'
```

The optional cell below prints equivalent checks from the selected YAML config. It does not run them unless you set `run_ssh_access_check = True`.

In [ ]:
import shlex
import yaml

run_ssh_access_check = False  # set True only when you want this notebook to SSH to the VMs

with config_path.open(encoding="utf-8") as handle:
    access_config = yaml.safe_load(handle)

transport = access_config.get("transport", {})
if transport.get("type") != "ssh":
    print("Selected transport is not ssh; use LoomAI slice tooling for VM access checks.")
else:
    username = str(transport.get("username") or "")
    ssh_options = transport.get("ssh_options") or []
    if isinstance(ssh_options, str):
        ssh_options = shlex.split(ssh_options)
    else:
        ssh_options = [str(item) for item in ssh_options]
    ssh_key = str(transport.get("ssh_key") or "").strip()
    base = ["ssh", *ssh_options]
    if ssh_key:
        base.extend(["-i", str(Path(ssh_key).expanduser())])

    hosts = dict(transport.get("hosts") or {})
    for key, value in transport.items():
        if key.endswith("_host") and value:
            hosts[key[:-5]] = str(value)

    slice_cfg = access_config.get("slice", {})
    network = access_config.get("network", {})
    paths = access_config.get("paths", {})
    pcap = access_config.get("pcap_replay", {})
    source = access_config.get("source_mode")

    checks = []
    receiver_node = str(slice_cfg.get("receiver_node") or "receiver")
    receiver_host = hosts.get(receiver_node)
    if receiver_host:
        receiver_cmd = (
            "set -e; hostname; whoami; command -v python3; command -v tcpdump; "
            f"test -d {shlex.quote(str(paths.get('wibviz_src') or ''))}; "
            f"test -f {shlex.quote(str(paths.get('mapping_csv') or ''))}; "
            f"ip -brief addr show dev {shlex.quote(str(network.get('receiver_interface') or ''))}"
        )
        checks.append(("receiver", [*base, f"{username}@{receiver_host}", "bash -lc " + shlex.quote(receiver_cmd)]))

    sender_node = str(slice_cfg.get("sender_node") or "sender")
    sender_host = hosts.get(sender_node)
    if source == "pcap_replay" and sender_host:
        sender_cmd = (
            "set -e; hostname; whoami; command -v tcpreplay; "
            f"test -f {shlex.quote(str(pcap.get('sender_pcap') or ''))}; "
            f"ip -brief addr show dev {shlex.quote(str(network.get('sender_interface') or ''))}"
        )
        checks.append(("sender", [*base, f"{username}@{sender_host}", "bash -lc " + shlex.quote(sender_cmd)]))

    for label, cmd in checks:
        print(f"{label} SSH check:")
        print(" ".join(shlex.quote(str(part)) for part in cmd))
        if run_ssh_access_check:
            result = subprocess.run(cmd, text=True, capture_output=True, check=False)
            print(result.stdout)
            if result.stderr:
                print(result.stderr)
            if result.returncode != 0:
                raise RuntimeError(f"{label} SSH check failed with exit code {result.returncode}")
        print()

    if not checks:
        print("No SSH checks were generated. Confirm receiver_host/sender_host and slice node names in the config.")

## Runner Helper

All executable cells below call the same package CLI. Dry-run is local-only and safe: it parses config, writes a small manifest under the local output root, and prints planned commands. Smoke/full runs contact VMs through the selected transport.

In [ ]:
def run_pipeline(*, dry_run=False, extra_args=None):
    cmd = [
        sys.executable,
        str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
        "--config",
        str(config_path),
        "--source-mode",
        source_mode,
    ]
    if output_root_override:
        cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
    if enable_adaptive:
        cmd.append("--adaptive")
    if not validate_results:
        cmd.append("--no-validate")
    if extra_args:
        cmd.extend(str(arg) for arg in extra_args)
    if dry_run:
        cmd.append("--dry-run")

    print("Running:", " ".join(str(part) for part in cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, check=False)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"runner failed with exit code {result.returncode}")
    return json.loads(result.stdout)

## Step 6: Run Dry-Run / Preflight Plan

Run this before any VM-contacting action. Dry-run does not run `tcpdump`, does not run `tcpreplay`, and does not SSH to VMs. It confirms config parsing, transport selection, local output paths, remote paths, and planned commands.

For external/local Jupyter, this is the minimum test that should work before you have VM reachability details fully validated.

In [ ]:
dry_payload = run_pipeline(dry_run=True)
payload = dry_payload
print("Dry-run manifest:", dry_payload["local_paths"]["manifest_json"])
print("Selected transport:", dry_payload["transport"])

## Step 7: First Smoke Test

A bounded smoke test is the first VM-contacting pipeline run. It runs real preflight checks, starts capture/replay for a small amount of work, decodes the chunks, downloads the combined CSV, and renders visualizations.

For `pcap_replay`, `--max-live-chunks 2` caps replay to two fixed Queue v2 chunks. With the default 64 packets/chunk, this replays 128 packets. For `live_nic`, the same argument captures two rolling chunks.

Use this command from a terminal for the validated LoomAI path:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --max-live-chunks 2
```

Use this command from external/local Jupyter after copying and editing the SSH config:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --max-live-chunks 2
```

In the notebook, set `run_smoke_test = True` in Step 0 only after the dry-run paths, interfaces, SSH settings, access checks, and required VM files look correct.

In [ ]:
smoke_args = ["--max-live-chunks", str(smoke_chunks)]
smoke_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
    "--config",
    str(config_path),
    "--source-mode",
    source_mode,
    *smoke_args,
]
if output_root_override:
    smoke_cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
if enable_adaptive:
    smoke_cmd.append("--adaptive")
if not validate_results:
    smoke_cmd.append("--no-validate")

print("Smoke command:", " ".join(str(part) for part in smoke_cmd))
if run_smoke_test:
    if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
        raise RuntimeError("Copy and edit the SSH template before running a smoke test.")
    payload = run_pipeline(extra_args=smoke_args)
else:
    print("Not running smoke test. Set run_smoke_test = True in the selection cell to execute it.")

### What Success Looks Like

For the bounded smoke test, the manifest/pass-fail metrics should show:

- `pass = true`
- `raw_packet_drops = 0`
- `raw_chunk_preserved_count` equals the bounded chunk count (`smoke_chunks`, normally 2).
- `decode_completed_chunk_count` equals the bounded chunk count (`smoke_chunks`, normally 2).
- `missing_row_count = 0`
- `extra_row_count = 0`

The 2-chunk test is only a quick health check. The full replay processes all 640 packets for the packaged event configs.

## Live Chunk-by-Chunk Demo

This section adds a notebook-native live display path for the packaged runner. The final packaged run still writes `artifacts/combined.csv`, `visualizations/heatmap.png`, and `visualizations/stacked_waveform.png` after completion. The live display is different: it watches local decoded chunk CSVs under `/tmp/dune_fabric_live_demo/runs/<RUN_ID>/artifacts/decoded_chunks/` and re-renders cumulative heatmap plus ridgeline as each `chunk_###.csv` appears.

The default presentation path now matches the slide evidence: event 4245 full replay = 640 packets, chunk size = 64 packets, therefore full event = 10 chunks. Use `live_demo_mode = "full"` for the presentation demo. If you need a quick connectivity check, change it to `"smoke"` for the 2-chunk fallback.

Run order for a single Jupyter kernel:

1. Run this cell first to configure live demo.
2. Run the next cell to start the live viewer. It starts a background notebook watcher and returns immediately.
3. Run the backend cell to start `scripts/run_pipeline.py` as a background subprocess, or copy the printed terminal command into a terminal.
4. Run the final status cell after the backend completes.

The live plots preserve actual cumulative chunk logic: `chunk_000.csv`, then `chunk_000.csv + chunk_001.csv`, continuing through all available chunks. They do not apply an artificial left-to-right mask.

For the presentation demo, both panels show the full event wire range, matching the benchmark/original visualization style. The ridgeline uses the benchmark scale: baseline at wire_id plus 0.01 * pedestal-subtracted ADC deviation.

Chunks are packet batches, not tick/time slices.

Raw ADC includes a large electronics baseline; the live demo subtracts a per-wire median pedestal so activity is shown as ADC deviation around zero.


In [ ]:
from datetime import datetime, timezone
import shlex
import yaml

print("Run this cell first to configure live demo.")
live_config = yaml.safe_load(Path(config_path).read_text())
if live_config.get("event_id") != "4245":
    print("Warning: default live demo evidence is event 4245; selected config uses event_id=", live_config.get("event_id"))
if live_config.get("slice", {}).get("name") == "dune-udp-demo-0ba777":
    raise RuntimeError("Do not use dead slice dune-udp-demo-0ba777; use dune-udp-demo-rebuild-2.")

live_demo_mode = "full"  # "full" = 10 chunks / 640 packets; "smoke" = 2 chunks / 128 packets
live_smoke_chunks = 2
live_full_event_chunks = 10
live_chunk_packets = int(live_config.get("decode", {}).get("chunking", {}).get("chunk_packets", 64))
live_total_packets = int(live_config.get("pcap_replay", {}).get("packet_limit", live_full_event_chunks * live_chunk_packets))
live_max_chunks = live_full_event_chunks if live_demo_mode == "full" else live_smoke_chunks
if live_demo_mode not in {"full", "smoke"}:
    raise ValueError("live_demo_mode must be 'full' or 'smoke'")
live_expected_packets = min(live_total_packets, live_max_chunks * live_chunk_packets)
live_poll_seconds = 2.0

# Event 4245 presentation styling: fixed axes and pedestal-subtracted color scale prevent resizing between updates.
live_tick_range = (0, 2047)
live_full_event_wire_range = (0, 199)
live_wire_range = live_full_event_wire_range
live_color_limits = (-20, 50)
live_waveform_mode = "full_range"
live_waveform_wire_range = live_full_event_wire_range
live_waveform_sample_count = None
live_waveform_adc_scale = 0.01  # matches benchmark signal_gain

live_run_id = f"notebook-live-4245-{live_demo_mode}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
live_output_root = Path(output_root_override).expanduser() if output_root_override else Path(live_config.get("paths", {}).get("output_root", "/tmp/dune_fabric_live_demo")).expanduser()
live_run_dir = live_output_root / "runs" / live_run_id
live_decoded_chunk_dir = live_run_dir / "artifacts" / "decoded_chunks"

live_backend_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
    "--config",
    str(config_path),
    "--source-mode",
    source_mode,
    "--run-id",
    live_run_id,
    "--max-live-chunks",
    str(live_max_chunks),
]
if output_root_override:
    live_backend_cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
if enable_adaptive:
    live_backend_cmd.append("--adaptive")
if not validate_results:
    live_backend_cmd.append("--no-validate")

print("Live demo mode:", live_demo_mode)
print(f"Event 4245 full replay: {live_total_packets} packets / {live_chunk_packets} packets per chunk = {live_full_event_chunks} chunks")
print(f"Selected live run: {live_max_chunks} chunks, {live_expected_packets} packets")
print("Live RUN_ID:", live_run_id)
print("Live run directory:", live_run_dir)
print("Decoded chunk CSV directory:", live_decoded_chunk_dir)
print("Notebook backend command:", " ".join(shlex.quote(str(part)) for part in live_backend_cmd))
print("Terminal command:", " ".join(shlex.quote(str(part)) for part in live_backend_cmd))
print("For a quick fallback test, set live_demo_mode = 'smoke' and rerun this cell before starting backend.")


In [ ]:
print("Run this cell to start the live viewer.")
import importlib.util
import inspect

# Always load the helper from this package checkout, even after a kernel restart
# or after a previous stale import from another directory.
viewer_module_path = (PACKAGE_ROOT / "scripts" / "live_chunk_viewer.py").resolve()
if not viewer_module_path.is_file():
    raise FileNotFoundError(f"Missing local live viewer helper: {viewer_module_path}")
spec = importlib.util.spec_from_file_location("live_chunk_viewer", viewer_module_path)
live_chunk_viewer = importlib.util.module_from_spec(spec)
sys.modules["live_chunk_viewer"] = live_chunk_viewer
spec.loader.exec_module(live_chunk_viewer)
start_notebook_live_viewer = live_chunk_viewer.start_notebook_live_viewer
print("live_chunk_viewer file:", live_chunk_viewer.__file__)
print("start_notebook_live_viewer signature:", inspect.signature(start_notebook_live_viewer))

live_viewer = start_notebook_live_viewer(
    live_run_dir,
    poll_seconds=live_poll_seconds,
    label=f"event 4245 {live_run_id}",
    stop_when_complete=True,
    expected_total_chunks=live_max_chunks,
    chunk_packets=live_chunk_packets,
    total_packets=live_expected_packets,
    tick_range=live_tick_range,
    wire_range=live_wire_range,
    color_limits=live_color_limits,
    waveform_mode=live_waveform_mode,
    waveform_wire_range=live_waveform_wire_range,
    waveform_window_size=live_waveform_sample_count,
    waveform_adc_scale=live_waveform_adc_scale,
)
print("Live viewer started in a background thread. Now run the backend cell or the printed terminal command.")


In [ ]:
print("Run this cell or terminal command to start the backend.")
if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
    raise RuntimeError("Copy and edit the SSH template before running the backend.")
live_reports_dir = live_run_dir / "reports"
live_reports_dir.mkdir(parents=True, exist_ok=True)
live_backend_stdout_path = live_reports_dir / "notebook_backend_stdout.log"
live_backend_stderr_path = live_reports_dir / "notebook_backend_stderr.log"
live_backend_stdout_handle = live_backend_stdout_path.open("w", encoding="utf-8", buffering=1)
live_backend_stderr_handle = live_backend_stderr_path.open("w", encoding="utf-8", buffering=1)
live_backend_process = subprocess.Popen(
    live_backend_cmd,
    cwd=str(PACKAGE_ROOT),
    text=True,
    stdout=live_backend_stdout_handle,
    stderr=live_backend_stderr_handle,
)
print("Started backend PID:", live_backend_process.pid)
print("stdout log:", live_backend_stdout_path)
print("stderr log:", live_backend_stderr_path)


In [ ]:
print("Run this cell to show final validation and outputs.")
process = globals().get("live_backend_process")
if process is not None:
    rc = process.wait()
    for handle_name in ("live_backend_stdout_handle", "live_backend_stderr_handle"):
        handle = globals().get(handle_name)
        if handle and not handle.closed:
            handle.close()
    print("Backend exit code:", rc)
else:
    rc = None
    print("No notebook backend process found; reading run directory status only.")

summary_path = live_run_dir / "reports" / "summary.json"
manifest_path = live_run_dir / "reports" / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found yet: {manifest_path}")
manifest = json.loads(manifest_path.read_text())
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
print(json.dumps({
    "run_id": manifest.get("run_id"),
    "status": manifest.get("status"),
    "phase": manifest.get("phase"),
    "pass": manifest.get("pass"),
    "validation_result": manifest.get("validation_result"),
    "queue_metrics": manifest.get("queue_metrics"),
    "expected_chunks": live_max_chunks,
    "expected_packets": live_expected_packets,
    "expected_rows_for_full_event": 409600 if live_demo_mode == "full" else None,
    "decoded_chunk_csvs": sorted(str(path) for path in live_decoded_chunk_dir.glob("chunk_*.csv")),
    "summary": summary,
}, indent=2, sort_keys=True))

from IPython.display import HTML, Image, display
import base64
import html as html_module


def _png_data_uri(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode("ascii")


def display_live_image_pair(heatmap_path, waveform_path, progress_label=None):
    heatmap_path = Path(heatmap_path)
    waveform_path = Path(waveform_path)
    missing = [path for path in (heatmap_path, waveform_path) if not path.is_file()]
    label_suffix = f" — {html_module.escape(str(progress_label))}" if progress_label else ""
    if missing:
        for path in missing:
            print("missing", path)
        return False
    html = f"""
    <div style="display:flex;gap:10px;align-items:flex-start;justify-content:center;width:100%;max-width:1100px;margin:0 auto;background:#080b0f;padding:8px;box-sizing:border-box;overflow:hidden;">
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Heatmap{label_suffix}</div>
        <img src="{_png_data_uri(heatmap_path)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Ridgeline{label_suffix}</div>
        <img src="{_png_data_uri(waveform_path)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
    </div>
    """
    display(HTML(html))
    return True


live_heatmap = live_run_dir / "visualizations" / "live_chunks" / "live_heatmap.png"
live_waveform = live_run_dir / "visualizations" / "live_chunks" / "live_stacked_waveform.png"
print(live_heatmap)
print(live_waveform)
display_live_image_pair(
    live_heatmap,
    live_waveform,
    progress_label=(live_viewer["state"].get("last_result") or {}).get("progress_label") if "live_viewer" in globals() else None,
)

for image_path in [
    live_run_dir / "visualizations" / "heatmap.png",
    live_run_dir / "visualizations" / "stacked_waveform.png",
]:
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))
if rc not in (None, 0):
    raise RuntimeError(f"Backend failed with exit code {rc}. Check {live_backend_stderr_path}")


## Step 8: Run Full Replay

Use this only after dry-run and the bounded smoke test look correct. Full replay processes the configured `pcap_replay.packet_limit` or the whole source PCAP if no packet limit is set.

The packaged event 4245 baseline uses `packet_limit: 640`, producing 10 raw chunks at 64 packets/chunk and 409,600 decoded rows. Full unbounded `live_nic` capture is intentionally not started from this cell; use the CLI with `--follow` only when that is the intended workflow.

In [ ]:
full_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
    "--config",
    str(config_path),
    "--source-mode",
    source_mode,
]
if output_root_override:
    full_cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
if enable_adaptive:
    full_cmd.append("--adaptive")
if not validate_results:
    full_cmd.append("--no-validate")

print("Full replay command:", " ".join(str(part) for part in full_cmd))
if run_full_replay:
    if source_mode != "pcap_replay":
        raise RuntimeError("This full-run cell is for pcap_replay. For live_nic, use --max-live-chunks or --follow intentionally.")
    if transport_backend == "ssh" and config_path.name.endswith("_template.yaml"):
        raise RuntimeError("Copy and edit the SSH template before running a full replay.")
    payload = run_pipeline()
else:
    print("Not running full replay. Set run_full_replay = True in the selection cell to execute it.")

## Step 9: Inspect Validation Metrics

After dry-run, this shows planned paths. After smoke/full runs, it shows pass/fail, validation status, Queue v2 metrics, chunk counts, and the combined CSV path from `reports/manifest.json`.

In [ ]:
active_payload = globals().get("payload", globals().get("dry_payload"))
if active_payload is None:
    raise RuntimeError("Run the dry-run cell first.")

if active_payload.get("dry_run"):
    manifest_path = Path(active_payload["local_paths"]["manifest_json"])
else:
    manifest_path = Path(active_payload["manifest"])
manifest = json.loads(manifest_path.read_text())

summary = {
    "dry_run": bool(active_payload.get("dry_run")),
    "run_id": manifest.get("run_id"),
    "status": manifest.get("status"),
    "phase": manifest.get("phase"),
    "source_mode": manifest.get("source_mode"),
    "transport": manifest.get("transport"),
    "pass": manifest.get("pass"),
    "validation_result": manifest.get("validation_result"),
    "queue_metrics": manifest.get("queue_metrics"),
    "raw_chunk_count": len(manifest.get("raw_chunks", [])),
    "chunk_count": len(manifest.get("chunks", [])),
    "combined_csv": manifest.get("paths", {}).get("local_combined_csv"),
}
print(json.dumps(summary, indent=2, sort_keys=True))

## Step 10: View Heatmap And Stacked Waveform Outputs

A smoke/full run downloads the combined CSV locally and writes visualization files under the local run directory.

The two main outputs are:

- Heatmap: overview of ADC activity across wires and ticks.
- Ridgeline / wire trace: line-based view that makes pulse shape and timing easier to see.

Dry-run has no CSV yet, so this cell reports the expected visualization directory but displays nothing.

In [ ]:
from IPython.display import HTML, display
import base64
import html as html_module

active_payload = globals().get("payload", globals().get("dry_payload"))
if active_payload is None:
    raise RuntimeError("Run the dry-run cell first.")

if active_payload.get("dry_run"):
    viz_dir = Path(active_payload["local_paths"]["visualization_dir"])
else:
    viz_dir = Path(active_payload["local_run_dir"]) / "visualizations"

heatmap = viz_dir / "heatmap.png"
waveform = viz_dir / "stacked_waveform.png"
print(f"Visualization directory: {viz_dir}")


def _png_data_uri(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode("ascii")


final_progress_label = None
try:
    manifest_chunks = active_payload.get("manifest", {}).get("chunks", [])
    packet_limit = active_payload.get("config", {}).get("pcap_replay", {}).get("packet_limit")
    chunk_packets = active_payload.get("config", {}).get("decode", {}).get("chunking", {}).get("chunk_packets", 64)
    if manifest_chunks and packet_limit:
        rendered_packets = min(len(manifest_chunks) * int(chunk_packets), int(packet_limit))
        expected_chunks = (int(packet_limit) + int(chunk_packets) - 1) // int(chunk_packets)
        final_progress_label = f"chunk {len(manifest_chunks)}/{expected_chunks}, {rendered_packets}/{int(packet_limit)} packets"
except Exception:
    final_progress_label = None
final_label_suffix = f" — {html_module.escape(str(final_progress_label))}" if final_progress_label else ""

if heatmap.exists() and waveform.exists():
    display(HTML(f"""
    <div style="display:flex;gap:10px;align-items:flex-start;justify-content:center;width:100%;max-width:1100px;margin:0 auto;background:#080b0f;padding:8px;box-sizing:border-box;overflow:hidden;">
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Heatmap{final_label_suffix}</div>
        <img src="{_png_data_uri(heatmap)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
      <div style="flex:0 1 49%;max-width:49%;min-width:0;text-align:center;color:#f3f6f9;font-family:sans-serif;box-sizing:border-box;">
        <div style="font-weight:700;margin:0 0 6px 0;">Ridgeline{final_label_suffix}</div>
        <img src="{_png_data_uri(waveform)}" style="width:100%;height:auto;object-fit:contain;display:block;border:1px solid #263442;border-radius:4px;" />
      </div>
    </div>
    """))
else:
    if not heatmap.exists():
        print("No heatmap available yet.")
    if not waveform.exists():
        print("No ridgeline available yet.")


## Step 11: Switch To A Different PCAP

For a different controlled replay input, start from `configs/custom_pcap_replay_template.yaml` or copy an SSH replay template and edit the same fields.

Minimum fields to review:

- `label` and `event_id`
- `pcap_replay.sender_pcap`, an absolute path on the sender VM
- `pcap_replay.packet_limit`, if you want a bounded full replay
- `decode.tpc` and `decode.plane`
- `paths.mapping_csv`, an absolute path on the receiver VM
- `validation.enabled`, `validation.reference_csv`, `validation.compare_fields`, `validation.expected_packets`, and `validation.expected_rows`

No Python code changes are required for a different PCAP if the packet format is supported by the existing WIBViz decoder.

## Step 12: Live NIC Mode / Real Data Path

Live NIC mode is the real-data direction. It skips sender-side replay and listens on the receiver NIC for packets that are already arriving from the detector/data path.

Use bounded chunks first:

```bash
python3 scripts/run_pipeline.py --config configs/live_nic_template.yaml --max-live-chunks 3
```

For SSH transport, copy and edit `configs/live_nic_ssh_template.yaml`, then dry-run before any real capture:

```bash
cp configs/live_nic_ssh_template.yaml configs/live_nic_ssh.yaml
python3 scripts/run_pipeline.py --config configs/live_nic_ssh.yaml --dry-run
```

Unbounded `--follow` mode is a prototype path. It should be used intentionally and stopped by creating `live_nic.stop_file` on the receiver VM.

## Optional: Repeated-Event Replay To Approximate Longer Live Data

Repeated replay is a controlled way to run the same PCAP multiple times. It can expose longer-run queue/decode behavior without needing a real continuous detector feed.

Important differences from real live data:

- It reuses the same event content, so detector variation and timing are artificial.
- Each repeat is a bounded `pcap_replay` run with its own run directory and manifest.
- Queue depth and decode lag are aggregated from per-repeat manifests; this is useful for smoke/performance checks, but it is not identical to one continuous live stream.

Dry-run a repeated replay plan without contacting VMs:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --repeat 2 --max-live-chunks 2 --dry-run
```

Short repeated replay smoke test after the normal 2-chunk smoke test already passes:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --repeat 2 --max-live-chunks 2
```

Alternating configs/events are supported by adding `--repeat-config`; the runner cycles through the config sequence:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --repeat-config configs/event_6860_replay.yaml --repeat 4 --max-live-chunks 2 --dry-run
```

The aggregate summary is written to `reports/repeated_replay_summary.json` under the repeated series run directory, normally below `/tmp/dune_fabric_live_demo/runs/<series_id>/`.

In [ ]:
# Optional repeated replay command builder. Safe default: print command only.
run_repeated_replay_dry_run = False
repeated_replay_repeat_count = 2
repeated_replay_smoke_chunks = 2
repeated_replay_extra_configs = []  # e.g. [PACKAGE_ROOT / "configs/event_6860_replay.yaml"]

repeated_cmd = [
    sys.executable,
    str(PACKAGE_ROOT / "scripts/run_pipeline.py"),
    "--config",
    str(config_path),
    "--source-mode",
    "pcap_replay",
    "--repeat",
    str(repeated_replay_repeat_count),
    "--max-live-chunks",
    str(repeated_replay_smoke_chunks),
    "--dry-run",
]
if output_root_override:
    repeated_cmd.extend(["--output-root", str(Path(output_root_override).expanduser())])
for extra_config in repeated_replay_extra_configs:
    repeated_cmd.extend(["--repeat-config", str(extra_config)])

print("Repeated replay dry-run command:", " ".join(str(part) for part in repeated_cmd))
if run_repeated_replay_dry_run:
    result = subprocess.run(repeated_cmd, text=True, capture_output=True, check=False)
    print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"repeated replay dry-run failed with exit code {result.returncode}")
else:
    print("Not running repeated replay dry-run. Set run_repeated_replay_dry_run=True to execute dry-run only.")

## Minimum External-Jupyter Test Path

Use this path when running from a laptop, desktop, or standard FABRIC notebook environment instead of LoomAI:

1. Copy the package to the local machine or clone the repository containing `dune_fabric_live_demo`.
2. Install local Python dependencies needed by the runner and notebook, including `pyyaml`, `pandas`, and `matplotlib`.
3. Confirm local command-line tools for the selected copy method: `ssh` and `scp`, or `ssh` and `rsync`.
4. Copy an SSH template: `cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml`.
5. Edit `transport.sender_host`, `transport.receiver_host`, `transport.username`, `transport.ssh_key` or `transport.ssh_options`, and all VM remote paths.
6. Run dry-run: `python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --dry-run`.
7. Only after real host/key/path details are correct, run a 2-chunk smoke test: `python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --max-live-chunks 2`.
8. Inspect local outputs under `/tmp/dune_fabric_live_demo/runs/<run_id>/` unless you set `--output-root`.

Keep this status honest: SSH transport is implemented and dry-run validated, but a true external-Jupyter smoke test still requires VM host/key details and VM-side files for a live slice.

## Common Setup Problems

Start with the dry-run JSON and `reports/manifest.json`. Confirm that local paths are on the Jupyter machine and remote paths are on the correct VM.

Common failures:

- `loomai` not found: you are probably outside LoomAI; use `transport.type: ssh` and an SSH config, or run from the LoomAI environment.
- SSH host/key failure: verify `sender_host`, `receiver_host`, `username`, `ssh_key`, and `ssh_options`; include `-F /path/to/fabric_ssh_config` or `-J bastion` if required.
- SSH command hangs: add `-o BatchMode=yes`, verify bastion routing, and test simple `ssh ... hostname` outside the notebook.
- Wrong local output location: remember `paths.output_root` is local to the Jupyter machine, not the receiver VM.
- `tcpdump` not found or permission denied: install it on the receiver and configure non-interactive capture permission.
- `tcpreplay` not found: install it on the sender for `pcap_replay`.
- Required file missing: check sender `pcap_replay.sender_pcap`, receiver `paths.mapping_csv`, receiver `paths.wibviz_src`, and optional receiver `validation.reference_csv`.
- Interface error: run `ip -brief addr` on the VM and update `network.sender_interface` or `network.receiver_interface`.
- No packets captured: re-check data-plane interface names, capture filter, sender/receiver IPs, and whether real traffic is present for `live_nic`.
- Decode fails: check `paths.wibviz_src`, `paths.mapping_csv`, and receiver-side Python dependencies.
- Validation fails: check that the reference CSV matches the selected PCAP and that `compare_fields` exist in both CSVs.

## Known Limitations / Not Automated Yet

- FABRIC slice creation and provisioning are not automated by this notebook.
- VM dependency installation is not fully automated.
- SSH host/key discovery is manual for external/local Jupyter.
- Input PCAP, mapping CSV, reference CSV, and WIBViz source staging are manual.
- Live continuous streaming exists as a bounded/follow capture path, not a polished production service.
- External SSH dry-run can validate config/planned commands without contacting VMs, but a real SSH smoke test requires actual host/key details and prepared VMs.

## Supporting References

This notebook is intended to be sufficient for a first run. These files remain useful when you need more detail:

- `README.md`: package overview and CLI examples.
- `docs/fresh_setup.md`: fresh-slice checklist.
- `docs/external_jupyter_setup.md`: external/local Jupyter SSH setup details.
- `docs/live_data_mode.md`: live NIC behavior and bounded vs follow mode.

## Command Reference

LoomAI dry-run, validated backend:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --dry-run
```

LoomAI smoke test, validated backend:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --max-live-chunks 2
```

LoomAI full replay:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml
```

External/local Jupyter SSH dry-run after copying and editing the template:

```bash
cp configs/event_4245_replay_ssh_template.yaml configs/event_4245_replay_ssh.yaml
python3 scripts/run_pipeline.py --config configs/event_4245_replay_ssh.yaml --dry-run
```

Custom PCAP without code changes:

```bash
cp configs/custom_pcap_replay_template.yaml configs/my_pcap.yaml
# edit label, pcap_replay.sender_pcap, decode filters, mapping CSV, and validation fields
python3 scripts/run_pipeline.py --config configs/my_pcap.yaml --dry-run
```

Repeated replay dry-run:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --repeat 2 --max-live-chunks 2 --dry-run
```

Repeated replay short smoke test:

```bash
python3 scripts/run_pipeline.py --config configs/event_4245_replay.yaml --repeat 2 --max-live-chunks 2
```


## Optional Tick Playback Helper

Decoded event playback by tick position, not live chunk order. This is separate from live packet chunk progress.


In [ ]:
def visual_playback_by_tick(source_image_path, label="Decoded event playback by tick position, not live chunk order"):
    """Placeholder hook for presentation-only tick-position playback.

    Live chunk mode intentionally renders cumulative packet batches in actual decoded
    chunk order. Use this helper name only for a separate presentation animation that
    reveals an already-decoded event by tick position; do not use it for live progress.
    """
    print(label)
    print("Live chunk visualizations do not mask left-to-right by tick.")
    return source_image_path
